# RIOs Growth-Rate Conversion

Public-safe Jupyter notebook distilled from a RHEED-intensity-oscillation workflow for converting oscillation-derived monolayer rates into growth-rate, composition, and timing estimates. Facility-specific operational text, private calibration files, and raw oscillation traces are intentionally excluded.

## Model

The source notebook used fitted relationships between effusion-cell temperature and oscillation-derived growth rate. In this public version, coefficients are placeholders and the workflow is the artifact: replace the fit coefficients with the current calibration before using the calculator for a real growth.

In [ ]:
import math
from dataclasses import dataclass

@dataclass(frozen=True)
class ArrheniusFit:
    slope: float
    intercept: float

def temperature_c_for_ml_rate(rate_ml_s, fit):
    """Return cell temperature in C for a target ML/s rate using a log-rate fit."""
    if rate_ml_s <= 0:
        raise ValueError("rate_ml_s must be positive")
    return (10000 * fit.slope) / (math.log(rate_ml_s) - fit.intercept) - 273.15

def ml_rate_for_temperature_c(temp_c, fit):
    """Return ML/s rate predicted by the same fit."""
    temp_k = temp_c + 273.15
    return math.exp((10000 * fit.slope) / temp_k + fit.intercept)

In [ ]:
# Placeholder coefficients for demonstration only.
fits = {
    "In": ArrheniusFit(slope=-2.6882, intercept=22.489),
    "Ga": ArrheniusFit(slope=-2.8119, intercept=21.349),
    "Al": ArrheniusFit(slope=-3.9053, intercept=26.722),
}

## Example: Ternary Target Split

For a target alloy, split the total group-III monolayer rate into component rates by composition. The numbers below are examples to illustrate the workflow.

In [ ]:
target_rate_ml_s = 0.16
in_fraction = 0.53
ga_fraction = 0.47

component_rates = {
    "In": target_rate_ml_s * in_fraction,
    "Ga": target_rate_ml_s * ga_fraction,
}

for element, rate in component_rates.items():
    temp_c = temperature_c_for_ml_rate(rate, fits[element])
    print(f"{element}: {rate:.4f} ML/s -> {temp_c:.1f} C")

## Timing From Thickness

Once a calibrated monolayer rate is chosen, convert a target thickness into shutter-open time. This simple calculation keeps the orientation and lattice-spacing assumptions explicit.

In [ ]:
def growth_time_seconds(thickness_nm, ml_rate_s, monolayer_spacing_nm):
    if thickness_nm <= 0 or ml_rate_s <= 0 or monolayer_spacing_nm <= 0:
        raise ValueError("all inputs must be positive")
    thickness_per_second = ml_rate_s * monolayer_spacing_nm
    return thickness_nm / thickness_per_second

# Example zinc-blende (111) monolayer spacing using a placeholder lattice constant.
lattice_constant_nm = 0.58687
monolayer_spacing_111_nm = lattice_constant_nm / math.sqrt(3)

target_thickness_nm = 200
time_s = growth_time_seconds(target_thickness_nm, target_rate_ml_s, monolayer_spacing_111_nm)
print(f"{target_thickness_nm} nm at {target_rate_ml_s:.3f} ML/s -> {time_s/60:.1f} min")

## Provenance Note

This public notebook is a cleaned derivative of an earlier RIOs calculator. It demonstrates the reproducible calculation pattern without publishing raw RHEED oscillation files or facility-specific procedures.